In [2]:
import glob
import os
import pandas as pd
import time
import sys

# Start total preprocessing timer
total_start = time.time()

# 1. Setup Folders
train_folder = "Training"
test_folder = "Testing"

# 2. Smart Training File Processing
start_time = time.time()
pre_concatenated_file = os.path.join(train_folder, "concatenated_dataset_Aug_2021_to_July_2024.csv")

if os.path.exists(pre_concatenated_file):
    pandas_train = pd.read_csv(pre_concatenated_file)
else:
    train_chunks = []
    for f in glob.glob(os.path.join(train_folder, "*_complete_data.csv")):
        train_chunks.append(pd.read_csv(f))
    for f in glob.glob(os.path.join(train_folder, "*.xlsx")):
        train_chunks.append(pd.read_excel(f))
    if not train_chunks:
        print("Error: No training files found!")
        exit()
    pandas_train = pd.concat(train_chunks, ignore_index=True)

# 3. Processing Testing Files
test_files = glob.glob(os.path.join(test_folder, "*.csv"))
if not test_files:
    print("Error: No testing files found!")
    exit()

test_list = [pd.read_csv(f) for f in test_files]
pandas_test = pd.concat(test_list, ignore_index=True)
ingestion_time = time.time() - start_time

# 4. Data Cleansing Tracking
initial_train_rows = len(pandas_train)
start_time = time.time()

pandas_train = pandas_train.dropna(subset=["main_aqi"]).drop_duplicates()
pandas_test = pandas_test.dropna(subset=["main_aqi"]).drop_duplicates()
cleaning_time = time.time() - start_time
final_train_rows = len(pandas_train)

# 5. Feature Engineering Tracking
start_time = time.time()
pandas_train['datetime'] = pd.to_datetime(pandas_train['datetime'], format='mixed', errors='coerce')
pandas_test['datetime'] = pd.to_datetime(pandas_test['datetime'], format='mixed', errors='coerce')

pandas_train = pandas_train.dropna(subset=['datetime'])
pandas_test = pandas_test.dropna(subset=['datetime'])

for target_df in [pandas_train, pandas_test]:
    target_df['hour'] = target_df['datetime'].dt.hour
    target_df['month'] = target_df['datetime'].dt.month
    target_df['year'] = target_df['datetime'].dt.year
feature_eng_time = time.time() - start_time

# Save the unified cleaned copy for Alisha's analytics visualizations
pandas_train.to_csv("clean_air_quality_for_plots.csv", index=False)
print("SUCCESS: 'clean_air_quality_for_plots.csv' has been created for Data Visualization!")

# Always generate the backup file for Maryum's fallback modeling step
pandas_test.to_csv("clean_air_quality_for_ml_testing.csv", index=False)

# 6. Spark Delivery Compiling Natively
spark_time = 0.0
spark_error_message = None

try:
    start_time = time.time()
    
    # Force environmental variable fallbacks inside python runtime to avoid system mismatches
    os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
    os.environ["SPARK_HOME"] = r"C:\Users\maryu\anaconda3\Lib\site-packages\pyspark"
    os.environ["HADOOP_HOME"] = r"C:\hadoop-3.4.1"
    
    import findspark
    findspark.init(os.environ["SPARK_HOME"])
    
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler

    spark = SparkSession.builder \
        .appName("DataEngineering") \
        .master("local[*]") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
        .getOrCreate()
    
    # Convert clean pandas dataframes to Spark contexts natively
    spark_train = spark.createDataFrame(pandas_train)
    spark_test = spark.createDataFrame(pandas_test)

    available_cols = spark_train.columns
    features_to_vectorize = [
        'components_co', 'components_no2', 'components_o3', 'components_pm2_5', 
        'components_pm10', 'components_so2', 'temperature_2m', 
        'relative_humidity_2m', 'wind_speed_10m', 'hour', 'month'
    ]
    
    input_features = [col for col in features_to_vectorize if col in available_cols]
    assembler = VectorAssembler(inputCols=input_features, outputCol="features")

    final_train_data = assembler.transform(spark_train).select("features", "main_aqi")
    final_test_data = assembler.transform(spark_test).select("features", "main_aqi")

    # Generate Parquet tables
    final_train_data.write.mode("overwrite").parquet("processed_train.parquet")
    print("SUCCESS: 'processed_train.parquet' has been created for Machine Learning Model Training!")
    
    final_test_data.write.mode("overwrite").parquet("processed_test.parquet")
    print("SUCCESS: 'processed_test.parquet' has been created for Machine Learning Model Testing!")
    
    spark.stop()
    spark_time = time.time() - start_time

except Exception as e:
    spark_error_message = str(e)
    print("Local Spark Parquet storage layout skipped or blocked by background Java environment setups.")

total_time = time.time() - total_start

# --- Final Metrics Printout ---
print("\n==================================================")
print("    PREPROCESSING TIME & PERFORMANCE METRICS")
print("==================================================")
print(f"Data Ingestion Time:       {ingestion_time:.2f} seconds")
print(f"Data Cleaning Time:        {cleaning_time:.2f} seconds")
print(f"Feature Engineering Time:  {feature_eng_time:.2f} seconds")
print(f"Spark Vectorization Time:  {spark_time:.2f} seconds")
print(f"Total Pipeline Duration:   {total_time:.2f} seconds")
print("--------------------------------------------------")
print("   DATA INTEGRITY & CLEANING ACCURACY")
print(f"Initial Training Rows:     {initial_train_rows}")
print(f"Cleaned Training Rows:     {final_train_rows}")
print(f"Dropped Corrupted Rows:    {initial_train_rows - final_train_rows}")
print("==================================================")
    
if spark_error_message:
    print(f"\nTechnical Spark Note:\n{spark_error_message}")
    print("==================================================")

SUCCESS: 'clean_air_quality_for_plots.csv' has been created for Data Visualization!
SUCCESS: 'processed_train.parquet' has been created for Machine Learning Model Training!
SUCCESS: 'processed_test.parquet' has been created for Machine Learning Model Testing!

    PREPROCESSING TIME & PERFORMANCE METRICS
Data Ingestion Time:       0.58 seconds
Data Cleaning Time:        0.25 seconds
Feature Engineering Time:  22.45 seconds
Spark Vectorization Time:  4.41 seconds
Total Pipeline Duration:   30.74 seconds
--------------------------------------------------
   DATA INTEGRITY & CLEANING ACCURACY
Initial Training Rows:     123134
Cleaned Training Rows:     123134
Dropped Corrupted Rows:    0
